# European Flight Delay Prediction — Raw Data Ingestion

This notebook downloads the original European Flight Delays 2023–2025 Parquet dataset from Hugging Face and uploads the unchanged raw file to Google Cloud Storage (GCS).


## Prerequisites

 The project dependencies are defined in the repository-level `requirements.txt`
file. When this notebook is opened in Google Colab from GitHub, the repository is cloned into the Colab environment
before installing the required dependencies.


In [ ]:
import os
# Clone the project repository into the Colab environment
REPO_URL = "https://github.com/sharonjmaye/european-flight-delay-prediction.git"
REPO_DIR = "european-flight-delay-prediction"
if os.path.exists(REPO_DIR):
    print(f"Repository already exists: {REPO_DIR}")
else:
    !git clone {REPO_URL} {REPO_DIR}
    print("Repository cloned successfully.")

# Change the current working directory to the project root
%cd {REPO_DIR}

## 1. Setup & Environment Installation

The project dependencies should be listed in `requirements.txt`. Run the following cell before continuing. If packages are newly installed, restart the Jupyter kernel if prompted.


In [ ]:
%pip install -r requirements.txt


## 2. Configuration & GCP Authentication

Set the project and bucket that the pipeline will use & authenticate to GCP.


In [ ]:
# --- GCP Configuration ---
PROJECT_ID = "setu-data-handling"
BUCKET_NAME = "european-flight-delay-prediction"
LOCATION = "EU"
# Source data  on Hugging Fataseace
RAW_URL = (
    "https://huggingface.co/datasets/345rf4gt56t4r3e3/"
    "flight-delays-europe-2023-2025/resolve/main/"
    "merged_flights_with_delay_2023_2025.parquet"
)

# Temporary local file in the Colab environment
LOCAL_RAW_PATH = "/content/merged_flights_with_delay_2023_2025.parquet"

# Destination inside your GCS bucket
GCS_RAW_PATH = "raw/merged_flights_with_delay_2023_2025.parquet"

from google.colab import auth
auth.authenticate_user()

print(f"Target project: {PROJECT_ID}")
print(f"Target bucket: gs://{BUCKET_NAME}")

## 3. Create the GGS Bucket if it doesn't already exist


In [ ]:
from google.cloud import storage
from google.api_core import exceptions


def create_gcs_bucket(bucket_name, project_id, location="EU"):
    """
    Create the GCS bucket if it does not already exist.

    Configuration:
      - EU multi-region
      - Standard storage
      - Uniform bucket-level access
      - Public read-only access
      - Object versioning enabled
      - Old object versions automatically cleaned up
      - Google-managed encryption
    """

    client = storage.Client(project=project_id)

    # Check whether the bucket already exists
    try:
        bucket = client.lookup_bucket(bucket_name)
        if bucket:
            print(f"Bucket already exists: gs://{bucket_name}")
            return bucket
    except (exceptions.NotFound, exceptions.Forbidden) as e:
        print(f"Bucket not accessible or doesn't exist yet: {e}")

    # ---------------------------------------------------------
    # 1. Configure and create bucket
    # ---------------------------------------------------------
    bucket = client.bucket(bucket_name)

    bucket.storage_class = "STANDARD"
    bucket.iam_configuration.uniform_bucket_level_access_enabled = True

    try:
        bucket = client.create_bucket(
            bucket,
            location=location
        )
    except exceptions.Forbidden as e:
        print("\n[ERROR] Creation Failed: You do not have permissions to create a bucket with this name in this project.")
        print("Please verify that your authenticated GCP user has 'Storage Admin' permissions on the project, and that the bucket name is globally unique.\n")
        raise e

    # ---------------------------------------------------------
    # 2. Enable object versioning
    # ---------------------------------------------------------
    bucket.versioning_enabled = True
    bucket.patch()

    # ---------------------------------------------------------
    # 3. Configure lifecycle rules for old object versions
    # ---------------------------------------------------------

    # Delete a noncurrent object when 5 newer versions exist
    bucket.add_lifecycle_delete_rule(
        num_newer_versions=5
    )

    # Delete a noncurrent object after 7 days
    bucket.add_lifecycle_delete_rule(
        days_since_noncurrent_time=7
    )

    bucket.patch()

    # ---------------------------------------------------------
    # 4. Grant public read-only access
    # ---------------------------------------------------------
    policy = bucket.get_iam_policy(requested_policy_version=3)

    policy.bindings.append({
        "role": "roles/storage.objectViewer",
        "members": {"allUsers"}
    })

    bucket.set_iam_policy(policy)

    # ---------------------------------------------------------
    # 5. Confirmation
    # ---------------------------------------------------------
    print(f"Successfully created bucket: gs://{bucket.name}")
    print(f"Location: {bucket.location}")
    print(f"Storage class: {bucket.storage_class}")
    print("Uniform bucket-level access: Enabled")
    print("Public access: Read-only")
    print("Object versioning: Enabled")
    print("Lifecycle: Delete noncurrent version after 5 newer versions")
    print("Lifecycle: Delete noncurrent version after 7 days")

    return bucket


create_gcs_bucket(
    BUCKET_NAME,
    PROJECT_ID,
    LOCATION
)

## 4. Download the raw Parquet file from Hugging Face

The file is streamed in chunks rather than loading the complete ~787 MB file into memory. No transformation is applied to the raw dataset.


In [ ]:
import requests
with requests.get(RAW_URL, stream=True, timeout=120) as response:
    response.raise_for_status()

    with open(LOCAL_RAW_PATH, "wb") as f:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)

file_size_mb = os.path.getsize(LOCAL_RAW_PATH) / (1024 * 1024)
print(f"Downloaded raw dataset locally: {LOCAL_RAW_PATH}")
print(f"File size: {file_size_mb:.2f} MB")


## 5. Upload the unchanged raw dataset to GCS


In [ ]:
def upload_to_gcs(bucket_name, source_file_name, destination_blob_name):
    """Upload a local file to a GCS object path."""
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(destination_blob_name)
    blob.upload_from_filename(source_file_name)

    print(
        f"Uploaded {source_file_name} -> "
        f"gs://{bucket_name}/{destination_blob_name}"
    )


upload_to_gcs(
    BUCKET_NAME,
    LOCAL_RAW_PATH,
    GCS_RAW_PATH
)


## 6. Verify the uploaded object

This confirms that the object exists in GCS and reports its stored size.


In [ ]:
client = storage.Client()
bucket = client.bucket(BUCKET_NAME)
blob = bucket.get_blob(GCS_RAW_PATH)

if blob is None:
    raise FileNotFoundError(f"Upload verification failed: gs://{BUCKET_NAME}/{GCS_RAW_PATH}")

print(f"Verified: gs://{BUCKET_NAME}/{GCS_RAW_PATH}")
print(f"GCS object size: {blob.size / (1024 * 1024):.2f} MB")


## Result

The original Parquet file is now retained in the GCS raw-data layer.

A separate processing/downsampling notebook can read from this raw object and create a reproducible 10K–100K working dataset without modifying the source file.
